In [1]:
%%capture
!pip install -U dspy pydantic dspy-ai

In [281]:
import dspy

api_key = ""
api_endpoint = "https://<endpoint>.services.ai.azure.com/"

lm = dspy.LM('azure/gpt-4.1', api_key = api_key, api_base=api_endpoint, api_version = '2024-10-21', max_tokens=4000)
dspy.configure(lm=lm)

In [282]:
patient_discharge_summary = """
Varón de 40 años trasladado a nuestro centro por una pérdida transitoria de conocimiento tras consumo de heroína. Presenta tumefacción de ambas extremidades izquierdas, con abolición de la motilidad y disminución de la sensibilidad de la extremidad inferior izquierda, ausencia de pulso pedio con persistencia de pulsos radial y cubital. En la analítica de ingreso presenta: CPK: 151.650 UI/l, GOT: 702 UI/l, GPT: 489 UI/l, creatinina: 4,1 mg/dl y potasio: 5,1 mmol/l. El cuadro resulta sospechoso de síndrome compartimental con rabdomiolisis e IRA secundaria, por lo que se decide realizar fasciotomías en quirófano de las extremidades izquierdas.
Desde el principio se intenta forzar diuresis con dopamina a dosis pre- ß, perfusión de furosemida a 0,1 mg/kg/h y manitol a 0,5 g/kg. Al tercer día se inicia hemodiálisis intermitente por el desarrollo de una insuficiencia renal anúrica con ascenso máximo de creatinina a 8,3 mg/dl con hiperpotasemia de 6,4 mmol/l. Progresivamente se va recuperando la función renal con creatinina al alta de la unidad de cuidados intensivos de 3,5 mg/dl y una diuresis espontánea de 3.900 cc/24h. El paciente desarrolló como secuela una afectación permanente sensitivo-motora del pie izquierdo, con una limitación de la motilidad flexo-extensora de los dedos y sensibilidad global hipoestésica-parestésica.
"""

In [4]:
signature = dspy.Signature(
        "patient_discharge_summary -> tox_habits: list[str]",
        instructions="You are an expert in clinical NLP in Spanish. Your task is to extract all toxic habits from the patient discharge summary as they appear in the Spanish text, like substance use and abuse. Make sure to retrieve all mentions of toxic habits.",
    )
full_text_cot = dspy.ChainOfThought(signature)
prediction = full_text_cot(patient_discharge_summary=patient_discharge_summary)
print(prediction)

Prediction(
    reasoning='En el resumen de alta se menciona explícitamente que el paciente tuvo una "pérdida transitoria de conocimiento tras consumo de heroína". Esto indica claramente el consumo de heroína como un hábito tóxico. No se mencionan otros hábitos tóxicos como consumo de alcohol, tabaco u otras drogas en el texto proporcionado.',
    tox_habits=['consumo de heroína']
)


In [5]:
from typing import Literal
from pydantic import BaseModel

class ToxHabit(BaseModel):
    tox_habit_text: str = dspy.OutputField(desc="All tokens referring to specific toxic habits terms that can be extracted from the discharge summary.")
    tox_habit_type: Literal['Tobacco', 'Cannabis', 'Alcohol', 'Drug'] = dspy.OutputField(desc="The type of the toxic habit.")

In [61]:
from typing import List
    
class ToxHabitExtraction(dspy.Signature):
    """
    You are an expert in clinical NLP in Spanish. 
    Extract contiguous tokens referring to specific toxic habits, like substance use and abuse, from a patient discharge summary as they appear in the Spanish text.
    Make sure to retrieve all mentions of toxic habits, if any.
    """
    patient_discharge_summary: str = dspy.InputField(desc="Patient discharge summary")
    #Tobacco, Cannabis, Alcohol and Drug - type
    tox_habits_tobacco: list[str] = dspy.OutputField(desc="All tobacco entities that can be extracted from the discharge summary.")
    tox_habits_cannabis: list[str] = dspy.OutputField(desc="All cannabis entities that can be extracted from the discharge summary.")
    tox_habits_alcohol: list[str] = dspy.OutputField(desc="All alcohol entities that can be extracted from the discharge summary.")
    tox_habits_drug: list[str] = dspy.OutputField(desc="All drug entities that can be extracted from the discharge summary.")

term_extractor = dspy.ChainOfThought(ToxHabitExtraction)

In [102]:
class ToxHabitTobaccoExtraction(dspy.Signature):
    """
    You are an expert in clinical NLP in Spanish. 
    Extract contiguous tokens referring to tobbaco use, from a patient discharge summary as they appear in the Spanish text.
    Make sure to retrieve all mentions of tobacco use, if any.
    """
    patient_discharge_summary: str = dspy.InputField(desc="Patient discharge summary")
    tobacco_entities: list[str] = dspy.OutputField(desc="All tobacco entities that can be extracted from the discharge summary.")
class ToxHabitCannabisExtraction(dspy.Signature):
    """
    You are an expert in clinical NLP in Spanish. 
    Extract contiguous tokens referring to cannabis use, from a patient discharge summary as they appear in the Spanish text.
    Make sure to retrieve all mentions of cannabis use, if any.
    """
    patient_discharge_summary: str = dspy.InputField(desc="Patient discharge summary")
    cannabis_entities: list[str] = dspy.OutputField(desc="All cannabis entities that can be extracted from the discharge summary.")
class ToxHabitAlcoholExtraction(dspy.Signature):
    """
    You are an expert in clinical NLP in Spanish. 
    Extract contiguous tokens referring to alcohol use and abuse, from a patient discharge summary as they appear in the Spanish text.
    Make sure to retrieve all mentions of alcohol use, if any.
    """
    patient_discharge_summary: str = dspy.InputField(desc="Patient discharge summary")
    alcohol_entities: list[str] = dspy.OutputField(desc="All alcohol entities that can be extracted from the discharge summary.")
class ToxHabitDrugExtraction(dspy.Signature):
    """
    You are an expert in clinical NLP in Spanish. 
    Extract contiguous tokens referring to drug abuse, from a patient discharge summary as they appear in the Spanish text.
    Make sure to retrieve all mentions of drug abuse, if any.
    """
    patient_discharge_summary: str = dspy.InputField(desc="Patient discharge summary")
    drug_entities: list[str] = dspy.OutputField(desc="All drug entities that can be extracted from the discharge summary.")

term_extractor_tobacco = dspy.ChainOfThought(ToxHabitTobaccoExtraction)
term_extractor_cannabis = dspy.ChainOfThought(ToxHabitCannabisExtraction)
term_extractor_alcohol = dspy.ChainOfThought(ToxHabitAlcoholExtraction)
term_extractor_drug = dspy.ChainOfThought(ToxHabitDrugExtraction)

In [103]:
term_extractor_drug(patient_discharge_summary=patient_discharge_summary)

Prediction(
    reasoning='En el resumen de alta, se menciona explícitamente que el paciente sufrió una pérdida transitoria de conocimiento tras consumo de heroína. Esta es una referencia clara y directa al abuso de una droga específica, en este caso, "heroína". No se mencionan otras sustancias de abuso en el texto. El resto de los fármacos mencionados (dopamina, furosemida, manitol) son utilizados como parte del tratamiento médico y no corresponden a abuso de drogas.',
    drug_entities=['heroína']
)

In [252]:
from typing import List
    
class PatientRecord(dspy.Signature):
    """
    You are an expert in clinical NLP in Spanish. 
    Extract contiguous tokens referring to specific toxic habits, like substance use and abuse, from a patient discharge summary as they appear in the Spanish text.
    Make sure to retrieve all phrases of toxic habits, if any.
    """
    
    patient_discharge_summary: str = dspy.InputField(desc="Patient discharge summary")
    #Tobacco, Cannabis, Alcohol and Drug - type
    tobacco_habits: list[str] = dspy.OutputField(desc="All tobacco entities that can be extracted from the discharge summary. For example: cigarrillos, tabaco, tabaquismo.")
    cannabis_habits: list[str] = dspy.OutputField(desc="All cannabis entities that can be extracted from the discharge summary. For example: marihuana, cannabis, hachís.")
    alcohol_habits: list[str] = dspy.OutputField(desc="All alcohol entities that can be extracted from the discharge summary. For example: enolísmo, alcohólica, cervezas.")
    drug_habits: list[str] = dspy.OutputField(desc="All drug entities that can be extracted from the discharge summary. For example: MDMA, drogas, otros tóxicos, abstinencia, otras sustancias adictivas.")

term_extractor_patient = dspy.ChainOfThought(PatientRecord)

In [256]:
from typing import Literal
from pydantic import BaseModel

class ToxHabit(BaseModel):
    toxic_habit_text: str = dspy.OutputField(desc="All tokens referring to specific toxic habits terms that can be extracted from the discharge summary.")
    #tox_habit_type: Literal['Tobacco', 'Cannabis', 'Alcohol', 'Drug'] = dspy.OutputField(desc="The type of the toxic habit.")
    toxic_use_substance: str = dspy.OutputField(desc="What kind of substance was used, for ex. cocaine, heroine, poppers")
    toxic_use_method: str = dspy.OutputField(desc="How was the substance used, for ex. intravenously, inhaled")
    toxic_use_amount: str = dspy.OutputField(desc="How much of the substance was used, for ex. 2 drinks, 10 cigarettes")
    toxic_use_frequency: str = dspy.OutputField(desc="How often was the substance used, for ex. for two years")
    toxic_use_duration: str = dspy.OutputField(desc="For how long was the substance used, for ex. for two year")
    toxic_use_history: str = dspy.OutputField(desc="Until was the substance used, for ex. in 2007")
    
class PatientDetailRecord(dspy.Signature):
    """
    You are an expert in clinical NLP in Spanish. 
    Extract contiguous tokens referring to specific toxic habits, like substance use and abuse, from a patient discharge summary as they appear in the Spanish text.
    Make sure to retrieve all phrases of toxic habits, if any. For each toxic substance, extract the details if any.
    """    
    patient_discharge_summary: str = dspy.InputField(desc="Patient discharge summary")
    #Tobacco, Cannabis, Alcohol and Drug - type
    tobacco_habits: list[ToxHabit] = dspy.OutputField(desc="All tobacco entities that can be extracted from the discharge summary. For example: cigarrillos, tabaco, tabaquismo.")
    cannabis_habits: list[ToxHabit] = dspy.OutputField(desc="All cannabis entities that can be extracted from the discharge summary. For example: marihuana, cannabis, hachís.")
    alcohol_habits: list[ToxHabit] = dspy.OutputField(desc="All alcohol entities that can be extracted from the discharge summary. For example: enolísmo, alcohólica, cervezas.")
    drug_habits: list[ToxHabit] = dspy.OutputField(desc="All drug entities that can be extracted from the discharge summary. For example: MDMA, drogas, otros tóxicos, abstinencia, otras sustancias adictivas.")

term_extractor_patient_details = dspy.ChainOfThought(PatientDetailRecord)

In [283]:
text = """
Mujer de casi 32 años, natural y residente en la zona, en seguimiento en nuestra UCA de Vinaròs desde los 18 años (en 2005); en terapia conmigo desde 2014 (año de mi incorporación a la plaza).
De acuerdo a las notas de la historia clínica de papel y diferentes documentos consultados para la sesión clínica -a menudo desordenados, informes de distinta procedencia y cotejo con apuntes propios-, la paciente se inició en el consumo de tabaco y alcohol a los 12 años, en el cannabis a los 13 (diario desde los 15), en la cocaína, speed, anfetaminas y éxtasis a los 15 también (ocio de los fines de semana, con alcohol), y años más tarde consumo de heroína -fumada y esnifada, mezclada con otras drogas- a los 26, aprox.
Lena dejó los estudios con 16 años, en 2º de la ESO, optando luego por trabajos muy precarios, erráticos, y sobre todo actividades marginales, entre ellas la prostitución hace 2 o 3 años, en Valencia.
Tiene reconocida una PNC (pensión no contributiva) por discapacidad del 73%, de la cual ella siempre se ha administrado el dinero, y hace un año fue inscrita por los Servicios Sociales de la localidad en un curso remunerado de administrativo, donde las condiciones de asistencia eran relativamente exigentes (horarios madrugadores, puntualidad, presencia, exposiciones), y a Lena le costaba bastante cumplir.
Mostraba esfuerzo, también motivada por el incentivo económico, pero había días que no acudía a clase, y ésta era también una de las razones para asistir con más frecuencia y compromiso a las citas médicas y psicológicas, que justificaban la ausencia de clase ese día.
Posteriormente ingresó voluntariamente en un centro de día para rehabilitación de tóxicos, con horarios poco compatibles con el curso de administrativo, sin perder la plaza gracias a una ILT (baja médica).
Respecto al entorno de Lena, la familia se caracteriza por su carencia de estructura.
Los padres de la paciente eran toxicómanos antes y durante su infancia, y ambos ya están fallecidos.
Ella fue acogida por la abuela materna, que también ha sido la persona que ha criado a una hermana 14 años menor, de un padre diferente (éste se encuentra vivo, también era toxicómano, fue presidiario, en la actualidad visita ocasionalmente a la familia, sobre todo a su hija, la hermana de Lena que ahora tiene 18 años).
Por tanto, desde el nacimiento la tutela de la paciente ha estado con los abuelos maternos, que actualmente cuentan con 68 años la mujer y 64 el marido (este hombre puede no ser el abuelo biológico de Lena, según una referencia de un informe de la historia clínica, y es la persona sobre la que actualmente ella deposita más hostilidad y rechazo).
La abuela es quien siempre se encarga de acompañar a Lena a los dispositivos, la ha rescatado en numerosas ocasiones de sitios hostiles y caóticos, ha supervisado muchas veces tratamientos y medicaciones, gestionado citas y visitas… es la principal figura de apego de la paciente, sin duda.
"""
term_extractor_patient(patient_discharge_summary=text)

Prediction(
    reasoning='En el resumen de alta se describen de manera explícita los hábitos tóxicos de la paciente desde la adolescencia. Se menciona el inicio del consumo de tabaco y alcohol a los 12 años, el consumo de cannabis a los 13 años (con uso diario desde los 15), y el consumo de otras drogas como cocaína, speed, anfetaminas y éxtasis a los 15 años, así como el consumo de heroína (fumada y esnifada, mezclada con otras drogas) a los 26 años aproximadamente. No se mencionan otros términos específicos relacionados con tóxicos distintos a los ya listados, ni se hace referencia a marihuana, hachís, ni a otras sustancias adictivas aparte de las mencionadas.',
    tobacco_habits=['tabaco'],
    cannabis_habits=['cannabis'],
    alcohol_habits=['alcohol'],
    drug_habits=['cocaína', 'speed', 'anfetaminas', 'éxtasis', 'heroína', 'otras drogas']
)

In [7]:
import random
from dspy.datasets import DataLoader
import pandas as pd

df_dataset = pd.read_csv('dataset.tsv', sep='\t')
df_dataset.head()

,filename,annotations,is_train,text
0,cc_habitos_toxicos454,"[{'trigger_type': 'Drug', 'trigger_text': 'her...",False,Varón de 40 años trasladado a nuestro centro p...
1,cc_habitos_toxicos603,"[{'trigger_type': 'Drug', 'trigger_text': 'cra...",True,Se presenta el caso de un paciente masculino d...
2,S1130-01082008000200010-1,"[{'trigger_type': 'Tobacco', 'trigger_text': '...",True,Paciente mujer de 74 años que ingresó por cuad...
3,cc_habitos_toxicos477,"[{'trigger_type': 'Drug', 'trigger_text': 'aya...",True,Se trata de un varón de 26 años procedente de ...
4,cc_habitos_toxicos326,"[{'trigger_type': 'Drug', 'trigger_text': 'tóx...",True,Anamnesis:\n\nEdad: 54 años\nSexo: masculino\n...


In [8]:
from ast import literal_eval
df_dataset['annotations'] = df_dataset['annotations'].apply(literal_eval)

In [9]:
df_dataset['response'] = ''
df_dataset['entities'] = ''

In [10]:
df_dataset_dev = df_dataset[~df_dataset['is_train']]

In [299]:
dev_dataset_text = []
train_dataset_text = []
for index, row in df_dataset.iterrows():
    items_list_tobacco = [ label['trigger_text'] for label in row['annotations'] if label['trigger_type'] == 'Tobacco']
    items_list_cannabis = [ label['trigger_text'] for label in row['annotations'] if label['trigger_type'] == 'Cannabis']
    items_list_alcohol = [ label['trigger_text'] for label in row['annotations'] if label['trigger_type'] == 'Alcohol']
    items_list_drug = [ label['trigger_text'] for label in row['annotations'] if label['trigger_type'] == 'Drug']
    example = dspy.Example(patient_discharge_summary=row['text'],
                filename=row['filename'],
                annotations=row['annotations'],
                tobacco_habits=items_list_tobacco,
                cannabis_habits=items_list_cannabis,
                alcohol_habits=items_list_alcohol,
                drug_habits=items_list_drug).with_inputs("patient_discharge_summary","filename","annotations")
    
    if row['is_train']:
        train_dataset_text.append(example)
    else:
        dev_dataset_text.append(example)

len(dev_dataset_text)

74

In [88]:
from dspy.teleprompt import LabeledFewShot

labeled_fewshot_optimizer = LabeledFewShot(k=3)
optimized_fewshot_term = labeled_fewshot_optimizer.compile(student = term_extractor, trainset=train_dataset_text, sample=True)
optimized_fewshot_term(patient_discharge_summary=patient_discharge_summary)

Prediction(
    reasoning='En el resumen de alta se menciona explícitamente "consumo de heroína" como el desencadenante del episodio de pérdida de conocimiento y de las complicaciones posteriores. No se hace ninguna mención a hábitos tóxicos relacionados con tabaco, cannabis o alcohol. Tampoco se mencionan otras drogas distintas a la heroína.',
    tox_habits_tobacco=[],
    tox_habits_cannabis=[],
    tox_habits_alcohol=[],
    tox_habits_drug=['consumo de heroína']
)

In [284]:
from tqdm import tqdm, tqdm_notebook

for index, row in tqdm(df_dataset_dev.iterrows(), total=df_dataset_dev.shape[0]):
    responses = []
    entities = []
    texts = row['text'].split('\n\n')
    for text in texts:
        response = term_extractor_patient(patient_discharge_summary=text) # term_extractor
        responses.append(response)
        current_entities = []
        for habit in response.tobacco_habits:
            current_entities.append({
                'trigger_type': 'Tobacco',
                'trigger_text': habit
            })
        for habit in response.alcohol_habits:
            current_entities.append({
                'trigger_type': 'Alcohol',
                'trigger_text': habit
            })
        for habit in response.cannabis_habits:
            current_entities.append({
                'trigger_type': 'Cannabis',
                'trigger_text': habit
            })
        for habit in response.drug_habits:
            current_entities.append({
                'trigger_type': 'Drug',
                'trigger_text': habit
            })
        entities.extend(current_entities)
    df_dataset_dev.at[index, 'response'] = responses
    df_dataset_dev.at[index, 'entities'] = entities

100%|███████████████████████████████████████████████████████████████████████████| 74/74 [07:31<00:00,  6.10s/it]


In [285]:
df_dataset_dev.head()

,filename,annotations,is_train,text,response,entities
0,cc_habitos_toxicos454,"[{'trigger_type': 'Drug', 'trigger_text': 'heroína', 'trigger_star...",False,Varón de 40 años trasladado a nuestro centro por una pérdida trans...,"[[reasoning, tobacco_habits, cannabis_habits, alcohol_habits, drug...","[{'trigger_type': 'Drug', 'trigger_text': 'consumo de heroína'}]"
21,caso_clinico_medicina_interna184,"[{'trigger_type': 'Tobacco', 'trigger_text': 'Fumadora', 'trigger_...",False,A.P: Tiroidectomía por bocio multinodular con hipotiroidismo secun...,"[[reasoning, tobacco_habits, cannabis_habits, alcohol_habits, drug...","[{'trigger_type': 'Tobacco', 'trigger_text': 'Fumadora activa 6 pa..."
30,cc_onco1765,"[{'trigger_type': 'Drug', 'trigger_text': 'hábitos tóxicos', 'trig...",False,"Anamnesis\nMujer de 59 años de edad, sin hábitos tóxicos. A los 40...","[[reasoning, tobacco_habits, cannabis_habits, alcohol_habits, drug...",[]
31,caso_clinico_urologia240,"[{'trigger_type': 'Tobacco', 'trigger_text': 'tabaco', 'trigger_st...",False,Anamnesis\nVarón de 71 años de edad que presenta como antecedentes...,"[[reasoning, tobacco_habits, cannabis_habits, alcohol_habits, drug...","[{'trigger_type': 'Tobacco', 'trigger_text': 'exfumador de un paqu..."
38,S0034-98872013000900015-1,"[{'trigger_type': 'Tobacco', 'trigger_text': 'tabaquismo', 'trigge...",False,"Paciente de 17 años, sexo femenino, con antecedentes de tabaquismo...","[[reasoning, tobacco_habits, cannabis_habits, alcohol_habits, drug...","[{'trigger_type': 'Tobacco', 'trigger_text': 'tabaquismo ocasional'}]"


In [286]:
result_filename = 'pred_dev_gpt41_lists_examples_per_field_temp0.5'

In [287]:
df_dataset_dev.to_csv(f'{result_filename}.tsv', sep='\t', index=False)

In [224]:
# extract all entities and properties at the same time?

In [274]:
def calculate_f1(gold_terms, pred_terms):
    gold_term_text = set([term.lower()  for term in gold_terms])
    pred_term_text = set([term.lower()  for term in pred_terms])

    true_positive = len(gold_term_text.intersection(pred_term_text))    
    false_positive = len(pred_term_text.difference(gold_term_text))
    false_negative = len(gold_term_text.difference(pred_term_text))

    denom =  (2*true_positive + false_positive + false_negative)
    f1 = 2 * true_positive/ (2*true_positive + false_positive + false_negative) if denom != 0 else 0
    
    return f1

In [376]:
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import scipy.sparse as sp
import typer

label_map = {
    'Tobacco': 1,
    'Alcohol': 2, 
    'Drug': 3,
    'Cannabis': 4
}
def iou_per_class(user_annotations: pd.DataFrame, target_annotations: pd.DataFrame) -> List[float]:
    """
    Calculate the IoU metric for each class in a set of annotations.
    """
    # Get mapping from note_id to index in array
    docs = np.unique(np.concatenate([user_annotations.filename, target_annotations.filename]))
    doc_index_mapping = dict(zip(docs, range(len(docs))))

    # Identify union of categories in GT and PRED
    cats = [0, 1, 2, 3, 4] #np.unique(np.concatenate([user_annotations.label, target_annotations.label]))

    # Find max character index in GT or PRED
    max_end = np.max(np.concatenate([user_annotations.off1, target_annotations.off1]))

    # Populate matrices for keeping track of character class categorization
    def populate_char_mtx(n_rows, n_cols, annot_df):
        mtx = sp.lil_array((n_rows, n_cols), dtype=np.uint64)
        for row in annot_df.itertuples():
            doc_index = doc_index_mapping[row.filename]
            mtx[doc_index, row.off0 : row.off1] = label_map[row.label]  # noqa: E203
        return mtx.tocsr()

    gt_mtx = populate_char_mtx(docs.shape[0], max_end, target_annotations)
    pred_mtx = populate_char_mtx(docs.shape[0], max_end, user_annotations)

    # Calculate IoU per category
    ious = []
    for cat in cats:
        gt_cat = gt_mtx == cat
        pred_cat = pred_mtx == cat
        # sparse matrices don't support bitwise operators, but the _cat matrices
        # have bool dtypes so when we multiply/add them we end up with only T/F values
        intersection = gt_cat * pred_cat
        union = gt_cat + pred_cat
        iou = intersection.sum() / union.sum() if union.sum() > 0 else 0
        ious.append(iou)

    return ious

In [377]:
import warnings
def calculate_metrics(gs, pred, subtask=['ner','norm']):
    '''       
    Calculate task Coding metrics:
    
    Two type of metrics are calculated: per document and micro-average.
    It is assumed there are not completely overlapping annotations.
    
    Parameters
    ---------- 
    gs : pandas dataframe
        with the Gold Standard. Columns are those defined in main function.
    pred : pandas dataframe
        with the predictions. Columns are those defined in main function.
    subtask : str
        subtask name
    
    Returns
    -------
    P_per_cc : pandas series
        Precision per clinical case (index contains clinical case names)
    P : float
        Micro-average precision
    R_per_cc : pandas series
        Recall per clinical case (index contains clinical case names)
    R : float
        Micro-average recall
    F1_per_cc : pandas series
        F-score per clinical case (index contains clinical case names)
    F1 : float
        Micro-average F1-score
    '''
    
    # Predicted Positives:
    Pred_Pos_per_cc = \
        pred.drop_duplicates(subset=['filename', "offset"]).\
        groupby("filename")["offset"].count()
    Pred_Pos = pred.drop_duplicates(subset=['filename', "offset"]).shape[0]

    # Gold Standard Positives:
    GS_Pos_per_cc = \
        gs.drop_duplicates(subset=['filename', "offset"]).\
        groupby("filename")["offset"].count()
    GS_Pos = gs.drop_duplicates(subset=['filename', "offset"]).shape[0]
    
    # Eliminate predictions not in GS (prediction needs to be in same clinical
    # case and to have the exact same offset to be considered valid!!!!)
    df_sel = pd.merge(pred, gs, 
                      how="right",
                      on=["filename", "offset", "label"])
    
    if subtask=='norm':
        # Check if codes are equal
        df_sel["is_valid"] = \
            df_sel.apply(lambda x: (x["code_x"] == x["code_y"]), axis=1)
    elif subtask=='ner':
        is_valid = df_sel.apply(lambda x: x.isnull().any()==False, axis=1)
        df_sel = df_sel.assign(is_valid=is_valid.values)
    else:
        raise Exception('Error! Subtask name not properly set up')

        
    # True Positives:
    TP_per_cc = (df_sel[df_sel["is_valid"] == True]
                 .groupby("filename")["is_valid"].count())
    TP = df_sel[df_sel["is_valid"] == True].shape[0]
    
    # Add entries for clinical cases that are not in predictions but are present
    # in the GS
    cc_not_predicted = (pred.drop_duplicates(subset=["filename"])
                        .merge(gs.drop_duplicates(subset=["filename"]), 
                              on='filename',
                              how='right', indicator=True)
                        .query('_merge == "right_only"')
                        .drop('_merge', axis=1))['filename'].to_list()
    for cc in cc_not_predicted:
        TP_per_cc[cc] = 0
    
    # Remove entries for clinical cases that are not in GS but are present
    # in the predictions
    cc_not_GS = (gs.drop_duplicates(subset=["filename"])
                .merge(pred.drop_duplicates(subset=["filename"]), 
                      on='filename',
                      how='right', indicator=True)
                .query('_merge == "right_only"')
                .drop('_merge', axis=1))['filename'].to_list()
    Pred_Pos_per_cc = Pred_Pos_per_cc.drop(cc_not_GS)

    # Calculate Final Metrics:
    P_per_cc =  TP_per_cc / Pred_Pos_per_cc 
    P = TP / Pred_Pos if Pred_Pos > 0 else 0
    R_per_cc = TP_per_cc / GS_Pos_per_cc
    R = TP / GS_Pos
    F1_per_cc = (2 * P_per_cc * R_per_cc) / (P_per_cc + R_per_cc)
    if (P+R) == 0:
        F1 = 0
        warnings.warn('Global F1 score automatically set to zero to avoid division by zero')
        return P_per_cc, P, R_per_cc, R, F1_per_cc, F1
    F1 = (2 * P * R) / (P + R)
    
    
    if ((any([F1, P, R]) > 1) | any(F1_per_cc>1) | any(P_per_cc>1) | any(R_per_cc>1) ):
        warnings.warn('Metric greater than 1! You have encountered an undetected bug, please, contact antonio.miranda@bsc.es!')
                                            
    return P_per_cc, P, R_per_cc, R, F1_per_cc, F1

In [378]:
def get_occurrences(term, text):
    occurrences = []
    i = 0
    while True:
    	f = text.find(term, i)
    	if f==-1:
    		break
    	occurrences.append(f)
    	i = f+1
    return occurrences

def get_entities(filename, text, term, label):
    entity_list = []
    if term.upper() not in text.upper():
        return entity_list
        
    indices = [(m, m+len(term)) for m in get_occurrences(term.upper(), text.upper())]
    for index in indices:
        start, end = index
        if term.upper() != text.upper()[start:end].upper():
            print(term, text[start:end])
        entity_list.append({
            'filename': filename,
            'mark': 'TOX',
            'label': label,
            'off0': start,
            'off1': end,
            'span': term
        })
    return entity_list

In [379]:
import numpy as np

def get_spans(example, pred):
    filename = example['filename']
    text = example['patient_discharge_summary']
    gold_annotations = example['annotations']
    
    predicted_entities = []
    predicted_spans = []
    for habit in pred.tobacco_habits:
        predicted_entities.append({
            'trigger_type': 'Tobacco',
            'trigger_text': habit
        })

    for habit in pred.alcohol_habits:
        predicted_entities.append({
            'trigger_type': 'Alcohol',
            'trigger_text': habit
        })

    for habit in pred.cannabis_habits:
        predicted_entities.append({
            'trigger_type': 'Cannabis',
            'trigger_text': habit
        })

    for habit in pred.drug_habits:
        predicted_entities.append({
            'trigger_type': 'Drug',
            'trigger_text': habit
        })

    for ent in predicted_entities:
        spans = get_entities(filename, text, ent['trigger_text'], ent['trigger_type'])
        predicted_spans.extend(spans)

    gold_spans = []

    for ent in gold_annotations:
        gold_spans.append({
            'filename': filename,
            'mark': 'TOX',
            'label': ent['trigger_type'],
            'off0': ent['trigger_start_span'],
            'off1': ent['trigger_end_span'],
            'span': ent['trigger_text'],
        })
    return gold_spans, predicted_spans

def f1(example, pred, trace=None): #entities only   
    gold_spans, predicted_spans = get_spans(example, pred)
        
    gs = pd.DataFrame.from_records(gold_spans, columns=['filename', 'mark', 'label', 'off0', 'off1', 'span'])
    pred = pd.DataFrame.from_records(predicted_spans, columns=['filename', 'mark', 'label', 'off0', 'off1', 'span'])

    gs['offset'] = gs['off0'].astype(str) + ' ' + gs['off1'].astype(str)
    pred['offset'] = pred['off0'].astype(str) + ' ' + pred['off1'].astype(str)
    #drop duplicates?
    P_per_cc, P, R_per_cc, R, F1_per_cc, F1 = calculate_metrics(gs, pred, subtask='ner')
    return F1


def iou_metric(example, pred, trace=None): #entities only
    gold_spans, predicted_spans = get_spans(example, pred)

    target_annotations = pd.DataFrame.from_records(gold_spans, columns=['filename', 'mark', 'label', 'off0', 'off1', 'span'])
    user_annotations = pd.DataFrame.from_records(predicted_spans, columns=['filename', 'mark', 'label', 'off0', 'off1', 'span'])
    ious = iou_per_class(user_annotations, target_annotations)
    
    return np.mean(ious)

In [384]:
def overlap_per_class(user_annotations: pd.DataFrame, target_annotations: pd.DataFrame) -> List[float]:
    """
    Calculate the IoU metric for each class in a set of annotations.
    """
    # Get mapping from note_id to index in array
    docs = np.unique(np.concatenate([user_annotations.filename, target_annotations.filename]))
    doc_index_mapping = dict(zip(docs, range(len(docs))))

    # Identify union of categories in GT and PRED
    cats = [1, 2, 3, 4] #np.unique(np.concatenate([user_annotations.label, target_annotations.label]))
    #cat_entities = [0,0,0,0]

    #for cat in label_map.keys():
    #    count = target_annotations[target_annotations.label==cat].shape[0]
    #    cat_entities[label_map[cat]-1] = count
    
    # Find max character index in GT or PRED
    max_end = np.max(np.concatenate([user_annotations.off1, target_annotations.off1]))

    # Populate matrices for keeping track of character class categorization
    def populate_char_mtx(n_rows, n_cols, annot_df):
        mtx = sp.lil_array((n_rows, n_cols), dtype=np.uint64)
        for row in annot_df.itertuples():
            doc_index = doc_index_mapping[row.filename]
            mtx[doc_index, row.off0 : row.off1] = label_map[row.label]  # noqa: E203])
        return mtx.tocsr()

    gt_mtx = populate_char_mtx(docs.shape[0], max_end, target_annotations)
    pred_mtx = populate_char_mtx(docs.shape[0], max_end, user_annotations)

    # Calculate IoU per category
    overlap_golds, overlap_preds = [], []
    for cat in cats:
        gt_cat = gt_mtx == cat
        pred_cat = pred_mtx == cat
        # sparse matrices don't support bitwise operators, but the _cat matrices
        # have bool dtypes so when we multiply/add them we end up with only T/F values
        intersection = gt_cat * pred_cat
        #union = gt_cat + pred_cat
        #iou = intersection.sum() / union.sum()
        overlap_gold = intersection.sum() / gt_cat.sum() if gt_cat.sum() > 0 else 0
        overlap_pred = intersection.sum() / pred_cat.sum() if pred_cat.sum() > 0 else 0
        overlap_golds.append(overlap_gold)
        overlap_preds.append(overlap_pred)

    return overlap_golds, overlap_preds

def overlap_metric(example, pred, trace=None): #entities only
    gold_spans, predicted_spans = get_spans(example, pred)

    target_annotations = pd.DataFrame.from_records(gold_spans, columns=['filename', 'mark', 'label', 'off0', 'off1', 'span'])
    user_annotations = pd.DataFrame.from_records(predicted_spans, columns=['filename', 'mark', 'label', 'off0', 'off1', 'span'])
    overlap_golds, overlap_preds = overlap_per_class(user_annotations, target_annotations)
    
    return np.mean(overlap_golds)

In [336]:
evaluate = dspy.Evaluate(devset=dev_dataset_text, metric=f1, num_threads=1, display_progress=True, display_table=5, provide_traceback=True)

In [ ]:
evaluate(term_extractor_patient) #18.55%

In [385]:
eval_overlap = dspy.Evaluate(devset=dev_dataset_text, metric=overlap_metric, num_threads=1, display_progress=True, display_table=5, provide_traceback=True)
eval_overlap(term_extractor_patient) #35.6% - includes empty class, 23.1% - w/o empty class

Average Metric: 17.13 / 74 (23.1%): 100%|██████████████████████████████████████| 74/74 [00:00<00:00, 197.81it/s]

2025/06/08 14:27:25 INFO dspy.evaluate.evaluate: Average Metric: 17.130821078431374 / 74 (23.1%)


,patient_discharge_summary,filename,annotations,example_tobacco_habits,example_cannabis_habits,example_alcohol_habits,example_drug_habits,reasoning,pred_tobacco_habits,pred_cannabis_habits,pred_alcohol_habits,pred_drug_habits,overlap_metric
0,Varón de 40 años trasladado a nuestro centro por una pérdida trans...,cc_habitos_toxicos454,"[{'trigger_type': 'Drug', 'trigger_text': 'heroína', 'trigger_star...",[],[],[],[heroína],"En el resumen de alta se menciona explícitamente el ""consumo de he...",[],[],[],[consumo de heroína],✔️ [0.250]
1,A.P: Tiroidectomía por bocio multinodular con hipotiroidismo secun...,caso_clinico_medicina_interna184,"[{'trigger_type': 'Tobacco', 'trigger_text': 'Fumadora', 'trigger_...",[Fumadora],[],[],[],"En el resumen de alta, se menciona explícitamente ""Fumadora activa...",[Fumadora activa 6 paquetes/años],[],[],[],✔️ [0.250]
2,"Anamnesis Mujer de 59 años de edad, sin hábitos tóxicos. A los 40 ...",cc_onco1765,"[{'trigger_type': 'Drug', 'trigger_text': 'hábitos tóxicos', 'trig...",[],[],[],[hábitos tóxicos],"En la anamnesis se menciona explícitamente ""sin hábitos tóxicos"", ...",[],[],[],[],
3,Anamnesis Varón de 71 años de edad que presenta como antecedentes ...,caso_clinico_urologia240,"[{'trigger_type': 'Tobacco', 'trigger_text': 'tabaco', 'trigger_st...",[tabaco],[],[],[],"En la anamnesis, se menciona que el paciente es ""exfumador de un p...",[exfumador de un paquete de tabaco al día durante 10 años],[],[],[],✔️ [0.250]
4,"Paciente de 17 años, sexo femenino, con antecedentes de tabaquismo...",S0034-98872013000900015-1,"[{'trigger_type': 'Tobacco', 'trigger_text': 'tabaquismo', 'trigge...",[tabaquismo],[],[],[],"En el resumen de alta, se menciona explícitamente ""antecedentes de...",[tabaquismo ocasional],[],[],[],✔️ [0.250]


23.15

In [382]:
eval_iou = dspy.Evaluate(devset=dev_dataset_text, metric=iou_metric, num_threads=1, display_progress=True, display_table=5, provide_traceback=True)

In [383]:
eval_iou(term_extractor_patient) #24.8% - includes empty class

/tmp/ipykernel_207292/845615941.py:69: SparseEfficiencyWarning: Comparing a sparse matrix with 0 using == is inefficient, try using != instead.
  ious = iou_per_class(user_annotations, target_annotations)


Average Metric: 18.33 / 74 (24.8%): 100%|██████████████████████████████████████| 74/74 [00:00<00:00, 173.53it/s]

2025/06/08 14:26:56 INFO dspy.evaluate.evaluate: Average Metric: 18.330208809628143 / 74 (24.8%)


,patient_discharge_summary,filename,annotations,example_tobacco_habits,example_cannabis_habits,example_alcohol_habits,example_drug_habits,reasoning,pred_tobacco_habits,pred_cannabis_habits,pred_alcohol_habits,pred_drug_habits,iou_metric
0,Varón de 40 años trasladado a nuestro centro por una pérdida trans...,cc_habitos_toxicos454,"[{'trigger_type': 'Drug', 'trigger_text': 'heroína', 'trigger_star...",[],[],[],[heroína],"En el resumen de alta se menciona explícitamente el ""consumo de he...",[],[],[],[consumo de heroína],✔️ [0.257]
1,A.P: Tiroidectomía por bocio multinodular con hipotiroidismo secun...,caso_clinico_medicina_interna184,"[{'trigger_type': 'Tobacco', 'trigger_text': 'Fumadora', 'trigger_...",[Fumadora],[],[],[],"En el resumen de alta, se menciona explícitamente ""Fumadora activa...",[Fumadora activa 6 paquetes/años],[],[],[],✔️ [0.214]
2,"Anamnesis Mujer de 59 años de edad, sin hábitos tóxicos. A los 40 ...",cc_onco1765,"[{'trigger_type': 'Drug', 'trigger_text': 'hábitos tóxicos', 'trig...",[],[],[],[hábitos tóxicos],"En la anamnesis se menciona explícitamente ""sin hábitos tóxicos"", ...",[],[],[],[],✔️ [0.145]
3,Anamnesis Varón de 71 años de edad que presenta como antecedentes ...,caso_clinico_urologia240,"[{'trigger_type': 'Tobacco', 'trigger_text': 'tabaco', 'trigger_st...",[tabaco],[],[],[],"En la anamnesis, se menciona que el paciente es ""exfumador de un p...",[exfumador de un paquete de tabaco al día durante 10 años],[],[],[],✔️ [0.149]
4,"Paciente de 17 años, sexo femenino, con antecedentes de tabaquismo...",S0034-98872013000900015-1,"[{'trigger_type': 'Tobacco', 'trigger_text': 'tabaquismo', 'trigge...",[tabaquismo],[],[],[],"En el resumen de alta, se menciona explícitamente ""antecedentes de...",[tabaquismo ocasional],[],[],[],✔️ [0.270]


24.77

In [340]:
from dspy.teleprompt import LabeledFewShot

labeled_fewshot_optimizer = LabeledFewShot(k=5)
optimized_fewshot = labeled_fewshot_optimizer.compile(student = term_extractor_patient, trainset=train_dataset_text)

In [341]:
evaluate(optimized_fewshot) #35.46%

Average Metric: 0.00 / 1 (0.0%):   1%|▌                                          | 1/74 [00:02<02:51,  2.35s/it]

/tmp/ipykernel_207292/1293182205.py:97: UserWarning: Global F1 score automatically set to zero to avoid division by zero
  warnings.warn('Global F1 score automatically set to zero to avoid division by zero')


Average Metric: 0.00 / 2 (0.0%):   3%|█▏                                         | 2/74 [00:04<02:29,  2.07s/it]

/tmp/ipykernel_207292/1293182205.py:97: UserWarning: Global F1 score automatically set to zero to avoid division by zero
  warnings.warn('Global F1 score automatically set to zero to avoid division by zero')


Average Metric: 0.00 / 3 (0.0%):   4%|█▋                                         | 3/74 [00:05<02:08,  1.81s/it]

...

/tmp/ipykernel_207292/1293182205.py:97: UserWarning: Global F1 score automatically set to zero to avoid division by zero
  warnings.warn('Global F1 score automatically set to zero to avoid division by zero')


Average Metric: 26.24 / 74 (35.5%): 100%|███████████████████████████████████████| 74/74 [08:17<00:00,  6.72s/it]

/tmp/ipykernel_207292/1293182205.py:97: UserWarning: Global F1 score automatically set to zero to avoid division by zero
  warnings.warn('Global F1 score automatically set to zero to avoid division by zero')
2025/06/08 13:59:36 INFO dspy.evaluate.evaluate: Average Metric: 26.23809523809524 / 74 (35.5%)


,patient_discharge_summary,filename,annotations,example_tobacco_habits,example_cannabis_habits,example_alcohol_habits,example_drug_habits,reasoning,pred_tobacco_habits,pred_cannabis_habits,pred_alcohol_habits,pred_drug_habits,f1
0,Varón de 40 años trasladado a nuestro centro por una pérdida trans...,cc_habitos_toxicos454,"[{'trigger_type': 'Drug', 'trigger_text': 'heroína', 'trigger_star...",[],[],[],[heroína],"En el resumen de alta se menciona explícitamente el ""consumo de he...",[],[],[],[consumo de heroína],
1,A.P: Tiroidectomía por bocio multinodular con hipotiroidismo secun...,caso_clinico_medicina_interna184,"[{'trigger_type': 'Tobacco', 'trigger_text': 'Fumadora', 'trigger_...",[Fumadora],[],[],[],"En el resumen de alta se menciona que la paciente es ""Fumadora act...",[Fumadora activa],[],[],[],
2,"Anamnesis Mujer de 59 años de edad, sin hábitos tóxicos. A los 40 ...",cc_onco1765,"[{'trigger_type': 'Drug', 'trigger_text': 'hábitos tóxicos', 'trig...",[],[],[],[hábitos tóxicos],En el resumen de alta se menciona explícitamente que la paciente e...,[],[],[],[],
3,Anamnesis Varón de 71 años de edad que presenta como antecedentes ...,caso_clinico_urologia240,"[{'trigger_type': 'Tobacco', 'trigger_text': 'tabaco', 'trigger_st...",[tabaco],[],[],[],"En la anamnesis se menciona que el paciente es ""exfumador de un pa...","[exfumador, tabaco]",[],[],[],✔️ [0.667]
4,"Paciente de 17 años, sexo femenino, con antecedentes de tabaquismo...",S0034-98872013000900015-1,"[{'trigger_type': 'Tobacco', 'trigger_text': 'tabaquismo', 'trigge...",[tabaquismo],[],[],[],En el resumen de alta se menciona que la paciente tiene antecedent...,[tabaquismo ocasional],[],[],[],


35.46

In [361]:
eval_iou(optimized_fewshot) #28.5%

/tmp/ipykernel_207292/845615941.py:69: SparseEfficiencyWarning: Comparing a sparse matrix with 0 using == is inefficient, try using != instead.
  ious = iou_per_class(user_annotations, target_annotations)


Average Metric: 21.06 / 74 (28.5%): 100%|██████████████████████████████████████| 74/74 [00:00<00:00, 140.81it/s]

2025/06/08 14:08:15 INFO dspy.evaluate.evaluate: Average Metric: 21.05509860752336 / 74 (28.5%)


,patient_discharge_summary,filename,annotations,example_tobacco_habits,example_cannabis_habits,example_alcohol_habits,example_drug_habits,reasoning,pred_tobacco_habits,pred_cannabis_habits,pred_alcohol_habits,pred_drug_habits,iou_metric
0,Varón de 40 años trasladado a nuestro centro por una pérdida trans...,cc_habitos_toxicos454,"[{'trigger_type': 'Drug', 'trigger_text': 'heroína', 'trigger_star...",[],[],[],[heroína],"En el resumen de alta se menciona explícitamente el ""consumo de he...",[],[],[],[consumo de heroína],✔️ [0.321]
1,A.P: Tiroidectomía por bocio multinodular con hipotiroidismo secun...,caso_clinico_medicina_interna184,"[{'trigger_type': 'Tobacco', 'trigger_text': 'Fumadora', 'trigger_...",[Fumadora],[],[],[],"En el resumen de alta se menciona que la paciente es ""Fumadora act...",[Fumadora activa],[],[],[],✔️ [0.250]
2,"Anamnesis Mujer de 59 años de edad, sin hábitos tóxicos. A los 40 ...",cc_onco1765,"[{'trigger_type': 'Drug', 'trigger_text': 'hábitos tóxicos', 'trig...",[],[],[],[hábitos tóxicos],En el resumen de alta se menciona explícitamente que la paciente e...,[],[],[],[],✔️ [0.182]
3,Anamnesis Varón de 71 años de edad que presenta como antecedentes ...,caso_clinico_urologia240,"[{'trigger_type': 'Tobacco', 'trigger_text': 'tabaco', 'trigger_st...",[tabaco],[],[],[],"En la anamnesis se menciona que el paciente es ""exfumador de un pa...","[exfumador, tabaco]",[],[],[],✔️ [0.250]
4,"Paciente de 17 años, sexo femenino, con antecedentes de tabaquismo...",S0034-98872013000900015-1,"[{'trigger_type': 'Tobacco', 'trigger_text': 'tabaquismo', 'trigge...",[tabaquismo],[],[],[],En el resumen de alta se menciona que la paciente tiene antecedent...,[tabaquismo ocasional],[],[],[],✔️ [0.250]


28.45

In [58]:
models = dict(prompt_model=lm, teacher_settings=dict(lm=lm))
tp = dspy.MIPROv2(metric=f1, auto="light", num_threads=16, **models)
kwargs = dict(minibatch_size=5, minibatch_full_eval_steps=2, requires_permission_to_run=False)
optimized_miprov2 = tp.compile(term_extractor, trainset=train_dataset_text[0:20], max_bootstrapped_demos=2, max_labeled_demos=2, **kwargs)

evaluate(optimized_miprov2) #13.6%

2025/06/05 11:26:12 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 16

2025/06/05 11:26:12 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/06/05 11:26:12 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/06/05 11:26:12 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


100%|█████████████████████████████████████████████████████████████████████████████| 4/4 [00:09<00:00,  2.46s/it]


Bootstrapped 1 full traces after 3 examples for up to 1 rounds, amounting to 4 attempts.
Bootstrapping set 4/6


 75%|█████████████████████████████████████████████████████████                   | 3/4 [00:00<00:00, 668.41it/s]


...

2025/06/05 11:33:03 INFO dspy.evaluate.evaluate: Average Metric: 2.2142857142857144 / 16 (13.8%)
2025/06/05 11:33:03 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 13.84 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 5'].
2025/06/05 11:33:03 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [7.5, 8.62, 8.07, 7.74, 7.5, 13.84, 8.07, 8.8, 8.31, 8.8, 13.84]
2025/06/05 11:33:03 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 13.84
2025/06/05 11:33:03 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/06/05 11:33:03 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 13.84!



Average Metric: 10.07 / 74 (13.6%): 100%|███████████████████████████████████████| 74/74 [06:14<00:00,  5.06s/it]

2025/06/05 11:39:18 INFO dspy.evaluate.evaluate: Average Metric: 10.066666666666666 / 74 (13.6%)


,patient_discharge_summary,example_tox_habits,reasoning,pred_tox_habits,f1
0,Varón de 40 años trasladado a nuestro centro por una pérdida trans...,"[{'term': 'heroína', 'label': 'Drug'}]",En el resumen de alta se menciona explícitamente que el paciente s...,[tox_habit_text='consumo de heroína' tox_habit_type='Drug'],
1,A.P: Tiroidectomía por bocio multinodular con hipotiroidismo secun...,"[{'term': 'Fumadora', 'label': 'Tobacco'}]",En el resumen de alta se menciona explícitamente que la paciente e...,[tox_habit_text='Fumadora activa 6 paquetes/años' tox_habit_type='...,
2,"Anamnesis Mujer de 59 años de edad, sin hábitos tóxicos. A los 40 ...","[{'term': 'hábitos tóxicos', 'label': 'Drug'}]","En la anamnesis se menciona explícitamente ""sin hábitos tóxicos"", ...",[],
3,Anamnesis Varón de 71 años de edad que presenta como antecedentes ...,"[{'term': 'tabaco', 'label': 'Tobacco'}]","En el resumen de alta se menciona que el paciente es ""exfumador de...",[tox_habit_text='exfumador de un paquete de tabaco al día durante ...,
4,"Paciente de 17 años, sexo femenino, con antecedentes de tabaquismo...","[{'term': 'tabaquismo', 'label': 'Tobacco'}]",En el resumen de alta se menciona explícitamente que la paciente t...,[tox_habit_text='tabaquismo ocasional' tox_habit_type='Tobacco'],


13.6

In [ ]:
#post processing

In [288]:
import re

entities_list = []

for index, row in df_dataset_dev.iterrows():
    if row['response'] == '':
        continue
        
    entities = row['entities']
    text = row['text']
    
    entity_list = []
    for ent in entities:
        term = ent[f'trigger_text']
        label = ent['trigger_type']

        entity_list.extend(get_entities(row['filename'], text, term, label))        
        
    entities_list.extend(entity_list)    

In [289]:
# resolve overlaps, take longest entity, use type of longest entity

In [290]:
import pandas as pd

df_entities_list = pd.DataFrame.from_records(entities_list)
df_entities_list.drop_duplicates(inplace=True)
df_entities_list.head()

,filename,mark,label,off0,off1,span
0,cc_habitos_toxicos454,TOX,Drug,94,112,consumo de heroína
1,caso_clinico_medicina_interna184,TOX,Tobacco,100,131,Fumadora activa 6 paquetes/años
2,caso_clinico_urologia240,TOX,Tobacco,89,145,exfumador de un paquete de tabaco al día durante 10 años
3,S0034-98872013000900015-1,TOX,Tobacco,56,76,tabaquismo ocasional
4,caso_clinico_endocrinologia68,TOX,Tobacco,262,296,Fumadora de 5/6 cigarrillos al día


In [291]:
df_entities_list[['filename','mark','label','off0','off1','span']].to_csv(f'{result_filename}_entities.tsv', sep='\t', index=False)

In [292]:
df_entities_list.shape

(105, 6)

In [27]:
# gold entities
gold_entities_list = []

for index, row in df_dataset_dev.iterrows():
    entities = row['annotations']
    text = row['text']
    
    entity_list = []
    for ent in entities:
        term = ent[f'trigger_text']
        label = ent['trigger_type']
        start = ent['trigger_start_span']
        end = ent['trigger_end_span']
        entity_list.append({
            'filename': row['filename'],
            'mark': 'TOX',
            'label': label,
            'off0': start,
            'off1': end,
            'span': term
        })
        
    gold_entities_list.extend(entity_list)

#'arguments': [{'filename': 'cc_habitos_toxicos454', 'trigger_type': 'Drug', 'trigger_text': 'heroína', 'trigger_start_span': 105, 'trigger_end_span': 112, 'argument_type': 'Type', 'argument_text': 'heroína', 'argument_start_span': 105, 'argument_end_span': 112, 'argument_subtype': ''}, {'filename': 'cc_habitos_toxicos454', 'trigger_type': 'Drug', 'trigger_text': 'heroína', 'trigger_start_span': 105, 'trigger_end_span': 112, 'argument_type': 'StatusTime', 'argument_text': 'consumo', 'argument_start_span': 94, 'argument_end_span': 101, 'argument_subtype': 'current'}]

In [28]:
df_gold_entities_list = pd.DataFrame.from_records(gold_entities_list)
df_gold_entities_list.head()

,filename,mark,label,off0,off1,span
0,cc_habitos_toxicos454,TOX,Drug,105,112,heroína
1,caso_clinico_medicina_interna184,TOX,Tobacco,100,108,Fumadora
2,cc_onco1765,TOX,Drug,40,55,hábitos tóxicos
3,caso_clinico_urologia240,TOX,Tobacco,116,122,tabaco
4,S0034-98872013000900015-1,TOX,Tobacco,56,66,tabaquismo


In [29]:
df_gold_entities_list[['filename','mark','label','off0','off1','span']].to_csv('gold_entities_tox.tsv', sep='\t', index=False)

In [ ]:
#iou

In [293]:
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import scipy.sparse as sp
import typer

label_map = {
    'Tobacco': 1,
    'Alcohol': 2, 
    'Drug': 3, 
    'Cannabis': 4
}

def iou_per_class(user_annotations: pd.DataFrame, target_annotations: pd.DataFrame) -> List[float]:
    """
    Calculate the IoU metric for each class in a set of annotations.
    """
    # Get mapping from note_id to index in array
    docs = np.unique(np.concatenate([user_annotations.filename, target_annotations.filename]))
    doc_index_mapping = dict(zip(docs, range(len(docs))))

    # Identify union of categories in GT and PRED
    cats = [1, 2, 3, 4] #np.unique(np.concatenate([user_annotations.label, target_annotations.label]))

    # Find max character index in GT or PRED
    max_end = np.max(np.concatenate([user_annotations.off1, target_annotations.off1]))

    # Populate matrices for keeping track of character class categorization
    def populate_char_mtx(n_rows, n_cols, annot_df):
        mtx = sp.lil_array((n_rows, n_cols), dtype=np.uint64)
        for row in annot_df.itertuples():
            doc_index = doc_index_mapping[row.filename]
            mtx[doc_index, row.off0 : row.off1] = label_map[row.label]  # noqa: E203
        return mtx.tocsr()

    gt_mtx = populate_char_mtx(docs.shape[0], max_end, target_annotations)
    pred_mtx = populate_char_mtx(docs.shape[0], max_end, user_annotations)

    # Calculate IoU per category
    ious = []
    for cat in cats:
        gt_cat = gt_mtx == cat
        pred_cat = pred_mtx == cat
        # sparse matrices don't support bitwise operators, but the _cat matrices
        # have bool dtypes so when we multiply/add them we end up with only T/F values
        intersection = gt_cat * pred_cat
        union = gt_cat + pred_cat
        iou = intersection.sum() / union.sum()
        ious.append(iou)

    return ious

user_annotations_path = f'{result_filename}_entities.tsv' # 'pred_dev_gpt41_entities.tsv' #0.45
# pred_dev_gpt41_separate_lists_entities = 0.4708
# pred_dev_gpt41_separate_lists_patient_entities 0.4631
# few shot 3 - 0.5020
# lists, examples per field - 0.5306, f1 - 0.44
target_annotations_path = 'gold_entities_tox.tsv'
user_annotations = pd.read_csv(user_annotations_path, sep='\t')
target_annotations = pd.read_csv(target_annotations_path, sep='\t')
ious = iou_per_class(user_annotations, target_annotations)
print(f"macro-averaged character IoU metric: {np.mean(ious):0.4f}")

macro-averaged character IoU metric: 0.3151


In [294]:
ious #tobbaco, alcohol, drug, cannabis

[0.32457496136012365, 0.32, 0.20393120393120392, 0.4117647058823529]

In [295]:
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import scipy.sparse as sp
import typer

label_map = {
    'Tobacco': 1,
    'Alcohol': 2, 
    'Drug': 3, 
    'Cannabis': 4
}

def overlap_per_class(user_annotations: pd.DataFrame, target_annotations: pd.DataFrame) -> List[float]:
    """
    Calculate the IoU metric for each class in a set of annotations.
    """
    # Get mapping from note_id to index in array
    docs = np.unique(np.concatenate([user_annotations.filename, target_annotations.filename]))
    doc_index_mapping = dict(zip(docs, range(len(docs))))

    # Identify union of categories in GT and PRED
    cats = [1, 2, 3, 4] #np.unique(np.concatenate([user_annotations.label, target_annotations.label]))
    cat_entities = [0,0,0,0]

    for cat in label_map.keys():
        count = target_annotations[target_annotations.label==cat].shape[0]
        cat_entities[label_map[cat]-1] = count
    
    # Find max character index in GT or PRED
    max_end = np.max(np.concatenate([user_annotations.off1, target_annotations.off1]))

    # Populate matrices for keeping track of character class categorization
    def populate_char_mtx(n_rows, n_cols, annot_df):
        mtx = sp.lil_array((n_rows, n_cols), dtype=np.uint64)
        for row in annot_df.itertuples():
            doc_index = doc_index_mapping[row.filename]
            mtx[doc_index, row.off0 : row.off1] = label_map[row.label]  # noqa: E203])
        return mtx.tocsr()

    gt_mtx = populate_char_mtx(docs.shape[0], max_end, target_annotations)
    pred_mtx = populate_char_mtx(docs.shape[0], max_end, user_annotations)

    # Calculate IoU per category
    overlap_golds, overlap_preds = [], []
    for cat in cats:
        gt_cat = gt_mtx == cat
        pred_cat = pred_mtx == cat
        # sparse matrices don't support bitwise operators, but the _cat matrices
        # have bool dtypes so when we multiply/add them we end up with only T/F values
        intersection = gt_cat * pred_cat
        #union = gt_cat + pred_cat
        #iou = intersection.sum() / union.sum()
        overlap_gold = intersection.sum() / gt_cat.sum()
        overlap_pred = intersection.sum() / pred_cat.sum()
        overlap_golds.append(overlap_gold)
        overlap_preds.append(overlap_pred)

    return overlap_golds, overlap_preds, gt_mtx, pred_mtx

user_annotations_path = f'{result_filename}_entities.tsv'
# few shot 3 - [0.9983417725267241, 0.7710843373493976, 0.5986622073578596, 1.0]
# [0.9995878905033962, 0.4, 0.28731942215088285, 0.4117647058823529]
# f'{result_filename}_entities.tsv' - lists_v2 - [0.9973147063039061, 0.8674698795180723, 0.5016722408026756, 1.0]
# [0.999324155934122,  0.34615384615384615, 0.24115755627009647, 0.32558139534883723]
# pred_dev_gpt41_separate_lists_patient_entities [0.9974453877867615, 0.8674698795180723, 0.4983277591973244, 1.0]
# [0.9993200238202835,  0.34615384615384615,  0.25254237288135595,  0.32558139534883723]
# pred_dev_gpt41_separate_lists_entities [0.9971503005674106, 1.0, 0.5317725752508361, 1.0]
# [0.9994084890274715, 0.3624454148471616, 0.2409090909090909, 0.32558139534883723]
#'pred_dev_gpt41_entities.tsv' [0.9977109662841774, 0.7710843373493976, 0.4414715719063545, 1.0]
#[0.9992147293138169, 0.3248730964467005, 0.2573099415204678, 0.32558139534883723]
target_annotations_path = 'gold_entities_tox.tsv'
user_annotations = pd.read_csv(user_annotations_path, sep='\t')
target_annotations = pd.read_csv(target_annotations_path, sep='\t')
overlap_golds, overlap_preds, gt_mtx, pred_mtx = overlap_per_class(user_annotations, target_annotations)
#print(f"macro-averaged character overlap metric: {np.mean(overlap):0.4f}")

In [296]:
overlap_golds #tobbaco, alcohol, drug, cannabis

[1.0, 0.7710843373493976, 0.5551839464882943, 1.0]

In [297]:
overlap_preds

[0.32457496136012365,
 0.35359116022099446,
 0.24375917767988253,
 0.4117647058823529]